<a href="https://colab.research.google.com/github/Shravani408/Netflix-Data-Cleaning-Using-Python-Description-/blob/Shravani408-patch-1/Netflix_Content_Rating_Genre_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from google.colab import files

In [2]:
uploaded = files.upload()

Saving netflix_cleaned.csv to netflix_cleaned.csv


In [5]:
df = pd.read_csv('netflix_cleaned.csv', encoding='latin-1')

print("Dataset Loaded Successfully!")

df.head()

Dataset Loaded Successfully!


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [6]:
print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nDataset Information:")
df.info()

Dataset Shape: (8790, 10)

Columns:
['show_id', 'type', 'title', 'director', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in']

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8790 non-null   object
 1   type          8790 non-null   object
 2   title         8790 non-null   object
 3   director      8790 non-null   object
 4   country       8790 non-null   object
 5   date_added    8790 non-null   object
 6   release_year  8790 non-null   int64 
 7   rating        8790 non-null   object
 8   duration      8790 non-null   object
 9   listed_in     8790 non-null   object
dtypes: int64(1), object(9)
memory usage: 686.8+ KB


In [7]:
rating_counts = df['rating'].fillna('Not Rated').value_counts()

print(rating_counts)

rating
TV-MA       3205
TV-14       2157
TV-PG        861
R            799
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64


In [8]:
fig = px.bar(
    x=rating_counts.index,
    y=rating_counts.values,
    title='Netflix Content Rating Distribution',
    labels={
        'x': 'Content Rating',
        'y': 'Number of Titles'
    },
    text=rating_counts.values
)

fig.update_layout(
    xaxis_title='Content Rating',
    yaxis_title='Number of Titles'
)

fig.show()

In [9]:
genre_df = df[['title', 'type', 'listed_in']].dropna().copy()

genre_df['listed_in'] = genre_df['listed_in'].str.split(',')

genre_df = genre_df.explode('listed_in')

genre_df['listed_in'] = genre_df['listed_in'].str.strip()

genre_df.head(10)

,title,type,listed_in
0,Dick Johnson Is Dead,Movie,Documentaries
1,Ganglands,TV Show,Crime TV Shows
1,Ganglands,TV Show,International TV Shows
1,Ganglands,TV Show,TV Action & Adventure
2,Midnight Mass,TV Show,TV Dramas
2,Midnight Mass,TV Show,TV Horror
2,Midnight Mass,TV Show,TV Mysteries
3,Confessions of an Invisible Girl,Movie,Children & Family Movies
3,Confessions of an Invisible Girl,Movie,Comedies
4,Sankofa,Movie,Dramas


In [10]:
genre_counts = genre_df['listed_in'].value_counts()

top_genres = genre_counts.head(15)

print("Top 15 Netflix Genres:")
print(top_genres)

Top 15 Netflix Genres:
listed_in
International Movies        2752
Dramas                      2426
Comedies                    1674
International TV Shows      1349
Documentaries                869
Action & Adventure           859
TV Dramas                    762
Independent Movies           756
Children & Family Movies     641
Romantic Movies              616
Thrillers                    577
TV Comedies                  573
Crime TV Shows               469
Kids' TV                     448
Docuseries                   394
Name: count, dtype: int64


In [11]:
fig = px.bar(
    x=top_genres.values,
    y=top_genres.index,
    orientation='h',
    title='Top 15 Most Common Netflix Genres',
    labels={
        'x': 'Number of Titles',
        'y': 'Genre'
    },
    text=top_genres.values
)

fig.update_layout(
    yaxis={'categoryorder': 'total ascending'}
)

fig.show()

In [12]:
rating_type = pd.crosstab(
    df['rating'].fillna('Not Rated'),
    df['type']
)

print(rating_type)

type      Movie  TV Show
rating                  
G            41        0
NC-17         3        0
NR           75        4
PG          287        0
PG-13       490        0
R           797        2
TV-14      1427      730
TV-G        126       94
TV-MA      2062     1143
TV-PG       540      321
TV-Y        131      175
TV-Y7       139      194
TV-Y7-FV      5        1
UR            3        0


In [13]:
rating_type_plot = rating_type.reset_index()

rating_type_melted = rating_type_plot.melt(
    id_vars='rating',
    var_name='Content Type',
    value_name='Count'
)

fig = px.bar(
    rating_type_melted,
    x='rating',
    y='Count',
    color='Content Type',
    barmode='group',
    title='Content Rating Comparison: Movies vs TV Shows'
)

fig.show()

In [14]:
genre_type = pd.crosstab(
    genre_df['listed_in'],
    genre_df['type']
)

top_10_genres = genre_type.sum(axis=1).nlargest(10).index

genre_comparison = genre_type.loc[top_10_genres]

print(genre_comparison)

type                      Movie  TV Show
listed_in                               
International Movies       2752        0
Dramas                     2426        0
Comedies                   1674        0
International TV Shows        0     1349
Documentaries               869        0
Action & Adventure          859        0
TV Dramas                     0      762
Independent Movies          756        0
Children & Family Movies    641        0
Romantic Movies             616        0


In [15]:
genre_comparison_plot = genre_comparison.reset_index()

genre_comparison_melted = genre_comparison_plot.melt(
    id_vars='listed_in',
    var_name='Content Type',
    value_name='Count'
)

fig = px.bar(
    genre_comparison_melted,
    x='listed_in',
    y='Count',
    color='Content Type',
    barmode='group',
    title='Top 10 Genres: Movies vs TV Shows'
)

fig.update_layout(
    xaxis_title='Genre',
    yaxis_title='Number of Titles',
    xaxis_tickangle=-45
)

fig.show()

In [16]:
print("========== NETFLIX AUDIENCE INSIGHTS ==========")

# Most common rating
most_common_rating = rating_counts.idxmax()
print("Most Common Rating:", most_common_rating)

# Most common genre
most_common_genre = genre_counts.idxmax()
print("Most Common Genre:", most_common_genre)

# Total Movies and TV Shows
content_type_counts = df['type'].value_counts()

print("\nContent Type Distribution:")
print(content_type_counts)

# Top 5 Genres
print("\nTop 5 Genres:")
print(genre_counts.head(5))

# Most common rating for each content type
print("\nMost Common Rating by Content Type:")

for content_type in ['Movie', 'TV Show']:
    subset = df[df['type'] == content_type]
    if not subset.empty:
        top_rating = (
            subset['rating']
            .fillna('Not Rated')
            .value_counts()
            .idxmax()
        )
        print(content_type, ":", top_rating)

========== NETFLIX AUDIENCE INSIGHTS ==========
Most Common Rating: TV-MA
Most Common Genre: International Movies

Content Type Distribution:
type
Movie      6126
TV Show    2664
Name: count, dtype: int64

Top 5 Genres:
listed_in
International Movies      2752
Dramas                    2426
Comedies                  1674
International TV Shows    1349
Documentaries              869
Name: count, dtype: int64

Most Common Rating by Content Type:
Movie : TV-MA
TV Show : TV-MA


In [17]:
rating_counts.to_csv(
    'Netflix_Rating_Analysis.csv',
    header=['Total Titles']
)

In [18]:
genre_counts.to_csv(
    'Netflix_Genre_Analysis.csv',
    header=['Total Titles']
)

In [20]:
rating_type.to_csv(
    'Netflix_Rating_Type_Comparison.csv'
)

In [21]:
genre_comparison.to_csv(
    'Netflix_Genre_Type_Comparison.csv'
)

In [22]:
files.download('Netflix_Rating_Analysis.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
files.download('Netflix_Genre_Analysis.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
files.download('Netflix_Rating_Type_Comparison.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
files.download('Netflix_Genre_Type_Comparison.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>